In [ ]:
import numpy as np

import multi_aster_spindle as mas
import fit_mt_length_histogram as fmlh

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import argparse
import os

ceph = '/mnt/home/ealca/ceph/'
home = '/mnt/home/ealca/'

# exp_dir = f'{ceph}multi_aster/useful/spindle_centring_10.0_lbar_5000_tubulin_0.0005_temp'
# exp_dir = f'{ceph}multi_aster/dt-L/single_aster_positioning_2500_tubulin_0.0005_temp'
# exp_dir = f'{ceph}multi_aster/useful/single_aster_positioning_reduced_tub_cost_1000_tubulin_0.002_temp'
exp_dir = f'{ceph}multi_aster/dt-L/uniform_13_dbar_10000_tubulin_0.0001_temp/'

spindle = mas.retrieve_experiement(experiment_dir=exp_dir, save_trajectory=True)

start_time = 125
end_time = 170

In [ ]:
def calculate_mt_lengths(mtoc_index, start_time, end_time):
    start_time = int(start_time / spindle.evolution_time)
    end_time = int(end_time / spindle.evolution_time)
    
    spindle_trace_path = os.path.join(spindle.dir_path, 'spindle_trace')
    
    # --- isolate the spindle states between start_time and end_time --- 
    
    # find the files and load their data
    top = (np.ceil(end_time / 1000) * 1000).astype(int) # nearest thousand greater than end_time
    bottom = (np.floor(start_time / 1000) * 1000).astype(int) # nearest thousand less than end_time
    
    temp_top = bottom + 1000
    temp_bottom = np.copy(bottom)
    
    temp_spindle_states = []
    
    while temp_top <= top:
        next_states_path = os.path.join(spindle_trace_path, f'pull_states_{temp_bottom}_{temp_top}.npy')
        temp_spindle_states.extend(np.load(next_states_path))
        # print(next_states_path)
        temp_bottom += 1000
        temp_top +=1000
    temp_spindle_states = np.array(temp_spindle_states)
    
    # -- cull this big set of spindle states to include only states between start_time and end_time --
    
    # find the number of states to cull from the start
    start_time_discrepancy = start_time - bottom
    end_time_discrepancy = top - end_time
    
    spindle_states_between_start_end = temp_spindle_states[start_time_discrepancy:]
    if end_time_discrepancy > 0:
        spindle_states_between_start_end = spindle_states_between_start_end[:-end_time_discrepancy]

    # # calculate and save the lengths of all MTs
    lengths = []
    for i in range(1, len(spindle_states_between_start_end)):
        pulling_states = spindle.pull_lattice[np.where(spindle_states_between_start_end[i] == mtoc_index)[0]]
        states_lengths = mas.normalize_vecs(pulling_states - spindle.mtoc_positions[mtoc_index])[1]

        lengths.extend(states_lengths)
    
    return lengths


# start_time = 100
# end_time = 500

pulling_mt_lengths = calculate_mt_lengths(1, start_time, end_time)
pulling_mt_lengths.extend(calculate_mt_lengths(2, start_time, end_time))

print(len(pulling_mt_lengths))

# print(f'mean MT length between {start_time}s and {end_time}s: {np.mean(pulling_mt_lengths)} um, std: {np.std(pulling_mt_lengths)}um')

# num_bins = 30
centers = np.arange(0, 21)          # 0, 1, 2, ..., 20
edges = centers - 0.5              # shift edges by half a bin width
edges = np.append(edges, 20.5)      # add the final right edge

fig, ax = plt.subplots()
ax.set_title(f'MT occupancy by length t={start_time}s and t={end_time}s\n Mean: {np.round(np.mean(pulling_mt_lengths),3)}um, std: {np.round(np.std(pulling_mt_lengths),3)}um')
ax.hist(pulling_mt_lengths, bins=edges)
ax.set_ylabel('count (thousands of seconds filled)')
ax.set_xlabel('MT length (um)')
plt.show()

# pulling_mt_lengths = calculate_mt_lengths(1, start_time, end_time)
# pulling_mt_lengths.extend(calculate_mt_lengths(2, start_time, end_time))

# print(len(pulling_mt_lengths))

In [ ]:
results = fmlh.analyze_exponential(pulling_mt_lengths, alpha=1.0, show_plots=True)

In [ ]:
def calculate_pulling_mt_lengths_at_birth(mtoc_index, start_time, end_time):
    start_time = int(start_time / spindle.evolution_time)
    end_time = int(end_time / spindle.evolution_time)
    
    spindle_trace_path = os.path.join(spindle.dir_path, 'spindle_trace')
    
    # --- isolate the spindle states between start_time and end_time --- 
    
    # find the files and load their data
    top = (np.ceil(end_time / 1000) * 1000).astype(int) # nearest thousand greater than end_time
    bottom = (np.floor(start_time / 1000) * 1000).astype(int) # nearest thousand less than end_time
    
    temp_top = bottom + 1000
    temp_bottom = np.copy(bottom)
    
    temp_spindle_states = []
    
    while temp_top <= top:
        next_states_path = os.path.join(spindle_trace_path, f'pull_states_{temp_bottom}_{temp_top}.npy')
        temp_spindle_states.extend(np.load(next_states_path))
        # print(next_states_path)
        temp_bottom += 1000
        temp_top +=1000
    temp_spindle_states = np.array(temp_spindle_states)
    
    # -- cull this big set of spindle states to include only states between start_time and end_time --
    
    # find the number of states to cull from the start
    start_time_discrepancy = start_time - bottom
    end_time_discrepancy = top - end_time
    
    spindle_states_between_start_end = temp_spindle_states[start_time_discrepancy:]
    if end_time_discrepancy > 0:
        spindle_states_between_start_end = spindle_states_between_start_end[:-end_time_discrepancy]
    
    # -- find birth lengths --
    
    times = np.array(list(spindle.trajectory.keys()))
    
    birth_lengths = []
    
    for i in range(1, len(spindle_states_between_start_end)):
        
        state_difference = (spindle_states_between_start_end[i] - spindle_states_between_start_end[i-1])
        
        if (state_difference == mtoc_index).any(): # states where new MTs impinge on motors
            # print(i)
            birth_index = np.where(state_difference == mtoc_index)[0][0] # known to be only one number
            time_of_birth = times[start_time + i]
            mtoc_pos_at_birth = spindle.trajectory[time_of_birth]['mtoc_pos'][mtoc_index]
            landing_site = spindle.pull_lattice[birth_index]
            mt_length_at_birth = mas.normalize_vecs(mtoc_pos_at_birth - landing_site)[1]
    
            birth_lengths.append(mt_length_at_birth)

    return np.array(birth_lengths)

# -- find the length of each MT at birth --
mtoc_index = 1

# loading files

pulling_lengths_at_birth = calculate_pulling_mt_lengths_at_birth(1, start_time, end_time)

print(f'mean MT length at birth: {np.mean(pulling_lengths_at_birth)}, std: {np.std(pulling_lengths_at_birth)}')

# num_bins = 40
centers = np.arange(0, 21)          # 0, 1, 2, ..., 20
edges = centers - 0.5              # shift edges by half a bin width
edges = np.append(edges, 20.5)      # add the final right edge

fig, ax = plt.subplots()
ax.set_title(f'MT lengths at birth between t={start_time} and t={end_time}\n Mean: {np.round(np.mean(pulling_lengths_at_birth),3)}, std: {np.round(np.std(pulling_lengths_at_birth),3)}')
ax.hist(pulling_lengths_at_birth, bins=edges)
ax.set_ylabel('count')
ax.set_xlabel('MT length (um)')
plt.show()


In [ ]:
results = fmlh.analyze_exponential(pulling_mt_lengths_at_birth, alpha=1.0, show_plots=True)

In [ ]:
# -- plot heatmap of motor occupancy
pull_lattice = spindle.pull_lattice

xs = pull_lattice[:,0]
ys = pull_lattice[:,1]
zs = pull_lattice[:,2]

# start_time = 55
# end_time = 140

occupancy = spindle.calculate_motor_occupancy(1, start_time, end_time)
norm = mcolors.Normalize(vmin=occupancy.min(), vmax=occupancy.max())

cmap = plt.cm.inferno
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
plt.title(f'occupancy proportion for MTOC 1 from {start_time} to {end_time} seconds')

ax.scatter(xs, ys, zs, c=occupancy,alpha=0.5, cmap=cmap, norm=norm)
ax.axis('equal')

mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable.set_array(occupancy)
fig.colorbar(mappable, ax=ax, shrink=0.5, label="Occupancy")
plt.show()

In [ ]:
def calculate_pulling_mt_lengths_at_death(mtoc_index, start_time, end_time):
    start_time = int(start_time / spindle.evolution_time)
    end_time = int(end_time / spindle.evolution_time)
    
    spindle_trace_path = os.path.join(spindle.dir_path, 'spindle_trace')
    
    # --- isolate the spindle states between start_time and end_time --- 
    
    # find the files and load their data
    top = (np.ceil(end_time / 1000) * 1000).astype(int) # nearest thousand greater than end_time
    bottom = (np.floor(start_time / 1000) * 1000).astype(int) # nearest thousand less than end_time
    
    temp_top = bottom + 1000
    temp_bottom = np.copy(bottom)
    
    temp_spindle_states = []
    
    while temp_top <= top:
        next_states_path = os.path.join(spindle_trace_path, f'pull_states_{temp_bottom}_{temp_top}.npy')
        temp_spindle_states.extend(np.load(next_states_path))
        # print(next_states_path)
        temp_bottom += 1000
        temp_top +=1000
    temp_spindle_states = np.array(temp_spindle_states)
    
    # -- cull this big set of spindle states to include only states between start_time and end_time --
    
    # find the number of states to cull from the start
    start_time_discrepancy = start_time - bottom
    end_time_discrepancy = top - end_time
    
    spindle_states_between_start_end = temp_spindle_states[start_time_discrepancy:]
    if end_time_discrepancy > 0:
        spindle_states_between_start_end = spindle_states_between_start_end[:-end_time_discrepancy]
    
    # -- find birth lengths --
    
    times = np.array(list(spindle.trajectory.keys()))
    
    death_lengths = []
    
    for i in range(1, len(spindle_states_between_start_end)):
        
        state_difference = (spindle_states_between_start_end[i] - spindle_states_between_start_end[i-1])
        
        if (state_difference == -mtoc_index).any(): # states where new MTs impinge on motors
            # print(i)
            death_index = np.where(state_difference == -mtoc_index)[0][0] # known to be only one number
            time_of_death = times[start_time + i]
            mtoc_pos_at_death = spindle.trajectory[time_of_death]['mtoc_pos'][mtoc_index]
            landing_site = spindle.pull_lattice[death_index]
            mt_length_at_death = mas.normalize_vecs(mtoc_pos_at_death - landing_site)[1]
    
            death_lengths.append(mt_length_at_death)

    return np.array(death_lengths)

# -- find the length of each MT at birth --
# mtoc_index = 1

# # loading files
# # start_time = 55
# # end_time = 119

# pulling_lengths_at_death = calculate_pulling_mt_lengths_at_death(1, start_time, end_time)



# print(f'mean MT length at birth: {np.mean(pulling_lengths_at_death)}, std: {np.std(pulling_lengths_at_death)}')

# num_bins = 40

# fig, ax = plt.subplots()
# ax.set_title(f'MT lengths at death between t={start_time} and t={end_time}\n Mean: {np.round(np.mean(pulling_lengths_at_death),3)}, std: {np.round(np.std(pulling_lengths_at_death),3)}')
# ax.hist(pulling_lengths_at_death, bins=num_bins)
# ax.set_ylabel('count')
# ax.set_xlabel('MT length (um)')
# plt.show()

In [ ]:
# ani_path = os.path.join(spindle.plot_folder_path, f'occupancy_ani.mp4')
# spindle.animate_mtoc_trajectory(save_path=ani_path, interval=50, stride=200, show_occupancy=True, occupancy_mtoc_id=1)

# spindle.optimize(10000000)

# spindle.plot_cost()